# 大语言模型后训练对齐 (RLHF / PPO / DPO / GRPO) 企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 大模型后训练 (Post-Training) / RL 对齐 / DeepSeek-R1 算法岗面试手撕  
> **核心涵盖**：GAE 广义优势估计、PPO-Clip 策略截断损失、Critic 价值裁剪、Schulman 逆向无偏 KL 散度、Bradley-Terry 奖励模型、DPO 偏好对损失与隐式奖励、GRPO 组内归一化相对优势、Token 级损失强化  
> **设计准则**：纯 PyTorch 逐行白盒手撕，全面覆盖 InstructGPT、Llama-3-Instruct、DeepSeek-Math/R1 工业级对齐基准。

---
### 核心模块速览
1. **模块一**：广义优势估计 (GAE, Generalized Advantage Estimation) 逆向时序手撕
2. **模块二**：PPO-Clip 策略截断损失手撕 (重要性比率 Ratio 与 Clip 保护边界)
3. **模块三**：Critic 价值网络均方误差损失与价值裁剪 (Value Clipping)
4. **模块四**：Schulman 逆向无偏 KL 散度估计手撕 (防方差爆炸)
5. **模块五**：Bradley-Terry 奖励模型 (Reward Model) 偏好对损失手撕
6. **模块六**：DPO (直接偏好优化) 核心损失函数手撕 (免奖励模型直连策略)
7. **模块七**：DPO 隐式奖励提取与动态胜率评估手撕
8. **模块八**：GRPO (群组相对策略优化) 组内均值方差归一化手撕 (彻底舍弃 Critic)
9. **模块九**：GRPO Token-Level 策略损失与相对优势联合更新

---
## 模块一：广义优势估计 (GAE, Generalized Advantage Estimation) 逆向时序手撕

### 【笔试考点与推导闭式】
1. **单步 TD 误差**：
   $$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$
2. **时序指数加权衰减**：
   $$A_t^{\text{GAE}} = \sum_{l=0}^\infty (\gamma \lambda)^l \delta_{t+l} = \delta_t + \gamma \lambda A_{t+1}^{\text{GAE}}$$
   从后向前逆序计算，时间复杂度严格为 $O(T)$！
3. **参数权衡**：$\lambda=0$ 退化为单步 TD（低方差高偏差）；$\lambda=1$ 退化为全蒙特卡洛（无偏差高方差）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_gae(rewards, values, next_value, gamma=0.99, lam=0.95):
    """
    广义优势估计 GAE 纯张量逆序手撕
    rewards: (T,) 各时间步即时奖励
    values: (T,) Critic 估计的当前状态价值
    next_value: 标量，最后一步的终止价值 V(s_T)
    """
    T = len(rewards)
    advantages = torch.zeros(T)
    last_adv = 0.0
    
    # 从最后一步逆向回推
    for t in reversed(range(T)):
        next_v = next_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_v - values[t]
        last_adv = delta + gamma * lam * last_adv
        advantages[t] = last_adv
        
    returns = advantages + values
    return advantages, returns

# 测试 GAE
r = torch.tensor([1.0, 0.5, 2.0])
v = torch.tensor([0.8, 1.2, 1.5])
v_next = 0.0
adv, ret = compute_gae(r, v, v_next, gamma=0.99, lam=0.95)
print("GAE 估计的 Advantage 优势值:", adv.numpy())
print("GAE 折扣回报 Return 目标值:", ret.numpy())
assert adv.shape == (3,) and ret.shape == (3,)
print(">>> GAE 广义优势估计验证通过！")

---
## 模块二：PPO-Clip 策略截断损失手撕 (重要性比率 Ratio 与 Clip 保护)

### 【笔试顶级高频考点】
- **概率比率 (Probability Ratio)**：
  $$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{old}}(a_t \mid s_t)} = \exp\left( \log \pi_\theta(a_t \mid s_t) - \log \pi_{\text{old}}(a_t \mid s_t) \right)$$
- **PPO-Clip 目标函数**：
  $$\mathcal{L}^{\text{CLIP}}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$
- **负对数目标转换为 Loss**：笔试优化器使用梯度下降，故损失函数取负号 $-\mathcal{L}^{\text{CLIP}}$！

In [ ]:
def ppo_clip_loss(log_probs, old_log_probs, advantages, eps_clip=0.2):
    """
    PPO-Clip 策略损失手撕
    """
    # 1. 计算概率比率 ratio
    ratio = torch.exp(log_probs - old_log_probs)
    
    # 2. 原始代理目标
    surr1 = ratio * advantages
    
    # 3. 截断代理目标
    surr2 = torch.clamp(ratio, 1.0 - eps_clip, 1.0 + eps_clip) * advantages
    
    # 4. 取悲观下界作为强化目标，加负号转为 Loss
    policy_loss = -torch.min(surr1, surr2).mean()
    return policy_loss, ratio

# 测试 PPO-Clip
cur_logp = torch.tensor([-0.2, -1.5, -0.5])
old_logp = torch.tensor([-0.25, -1.0, -0.5])
advs = torch.tensor([1.2, -0.8, 0.5])
p_loss, r_val = ppo_clip_loss(cur_logp, old_logp, advs, eps_clip=0.2)
print("PPO-Clip 策略 Loss:", p_loss.item())
print("概率比率 Ratio (部分被截断):", r_val.numpy())
assert not torch.isnan(p_loss)
print(">>> PPO-Clip 策略截断保护验证成功！")

---
## 模块三：Critic 价值网络均方误差损失与价值裁剪 (Value Clipping)

### 【笔试考点】
- 为防止 Critic 价值网络在单次更新中发生剧烈震荡，PPO 同样对价值预测施加类似 Clip：
  $$V_{\text{clipped}} = V_{\text{old}} + \text{clip}(V - V_{\text{old}}, -\epsilon, \epsilon)$$
  $$\mathcal{L}_V = \frac{1}{2} \max\left( (V - V_{\text{target}})^2, (V_{\text{clipped}} - V_{\text{target}})^2 \right)$$

In [ ]:
def critic_loss_clipped(v_pred, old_v_pred, returns, eps_clip=0.2):
    """
    带 Clip 的价值网络损失函数
    """
    # 原始 MSE
    loss_unclipped = (v_pred - returns) ** 2
    
    # 截断预测并计算 MSE
    v_clipped = old_v_pred + torch.clamp(v_pred - old_v_pred, -eps_clip, eps_clip)
    loss_clipped = (v_clipped - returns) ** 2
    
    # 取悲观最大值
    val_loss = 0.5 * torch.max(loss_unclipped, loss_clipped).mean()
    return val_loss

v_curr = torch.tensor([1.1, 1.8])
v_old = torch.tensor([1.0, 1.0])
rets = torch.tensor([1.5, 2.0])
v_loss = critic_loss_clipped(v_curr, v_old, rets, eps_clip=0.2)
print("Critic 价值网络裁剪 Loss:", v_loss.item())
assert v_loss.item() >= 0
print(">>> Critic 价值裁剪验证成功！")

---
## 模块四：Schulman 逆向无偏 KL 散度估计手撕 (防策略漂移与方差爆炸)

### 【笔试考点与推导】
1. **经典 KL 散度估计**：$D_{KL} = \log \frac{\pi_\theta}{\pi_{\text{ref}}} = \log \pi_\theta - \log \pi_{\text{ref}}$；
2. **痛点**：单样本估计极易产生负值，导致方差巨大；
3. **Schulman 逆向无偏估计公式 (John Schulman, 2020)**：
   $$D_{KL}^{\text{approx}} = \frac{\pi_{\text{ref}}}{\pi_\theta} - 1 - \log \frac{\pi_{\text{ref}}}{\pi_\theta} = r - 1 - \log r, \quad \text{其中 } r = \frac{\pi_{\text{ref}}}{\pi_\theta}$$
   利用不等式 $x - 1 - \ln x \ge 0$，**该单样本估计值恒严格非负**，且一阶导为 0、二阶导与真实 KL 完全一致，数值极其平稳！

In [ ]:
def schulman_unbiased_kl(log_pi, log_ref):
    """
    John Schulman 逆向无偏且严格非负的 KL 散度估计
    """
    # 比例比值 r = pi_ref / pi_curr
    ratio = torch.exp(log_ref - log_pi)
    # k3 估计量: r - 1 - log(r)
    kl = ratio - 1.0 - (log_ref - log_pi)
    return kl.mean()

log_p_curr = torch.tensor([-1.2, -0.8, -2.1])
log_p_ref = torch.tensor([-1.0, -0.9, -2.0])
kl_val = schulman_unbiased_kl(log_p_curr, log_p_ref)
print("Schulman 逆向无偏 KL 估计值:", kl_val.item())
assert kl_val.item() >= 0.0, "数学定理: Schulman KL 估计恒大于等于 0!"
print(">>> Schulman 逆向无偏 KL 验证通过！")

---
## 模块五：Bradley-Terry 奖励模型 (Reward Model) 偏好对损失手撕

### 【笔试考点与推导】
- **Bradley-Terry 模型**：胜出回答 $y_w$ 相对于落败回答 $y_l$ 的概率建模为：
  $$P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l))$$
- **损失函数**：
  $$\mathcal{L}_{\text{RM}} = - \mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma(r(x, y_w) - r(x, y_l)) \right]$$

In [ ]:
def bradley_terry_reward_loss(r_chosen, r_rejected):
    """
    r_chosen: (B,) 优质回答的标量打分
    r_rejected: (B,) 劣质回答的标量打分
    """
    # 差值过 sigmoid 取对数
    loss = -torch.log(torch.sigmoid(r_chosen - r_rejected) + 1e-8).mean()
    return loss

r_w = torch.tensor([3.5, 1.2]) # 好回答得分高
r_l = torch.tensor([1.0, 0.5]) # 差回答得分低
rm_loss = bradley_terry_reward_loss(r_w, r_l)
print("奖励模型 Pairwise 对比 Loss:", rm_loss.item())
assert rm_loss.item() > 0
print(">>> Bradley-Terry 奖励模型验证成功！")

---
## 模块六：DPO (直接偏好优化) 核心损失函数手撕 (免奖励模型)

### 【笔试压轴必背考点】
1. **DPO 突破**：彻底丢弃独立的奖励模型 (Reward Model) 与复杂四模型架构，直接将奖励解析代入最优策略，得到封闭解；
2. **闭式损失公式**：
   $$\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = - \mathbb{E} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right]$$
   - $\beta$：温度控制超参（通常取 0.1）。

In [ ]:
def dpo_loss(pi_chosen_logps, pi_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """
    DPO 闭式损失函数纯 PyTorch 手撕
    所有 logps 均为该回答全序列 Token 对数概率之和: (B,)
    """
    # 1. 计算当前策略与参考策略的对数比率 (即隐式奖励差)
    pi_logratios = pi_chosen_logps - pi_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps
    
    # 2. 偏好拔河 Logit 差值
    logits = beta * (pi_logratios - ref_logratios)
    
    # 3. 负对数 sigmoid 损失
    loss = -F.logsigmoid(logits).mean()
    return loss, logits

# 测试 DPO 损失
pi_w = torch.tensor([-2.0, -1.5])
pi_l = torch.tensor([-4.0, -3.0])
ref_w = torch.tensor([-2.5, -2.0])
ref_l = torch.tensor([-3.5, -2.5])

dpo_l, diff_logits = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1)
print("DPO 单步闭式 Loss:", dpo_l.item())
print("隐式相对胜出 Logit 差值:", diff_logits.numpy())
assert dpo_l.item() > 0
print(">>> DPO 损失手撕推演通过！")

---
## 模块七：DPO 隐式奖励提取与动态胜率评估手撕

### 【笔试考点】
- **隐式奖励公式**：
  $$r_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)} = \beta (\log \pi_\theta - \log \pi_{\text{ref}})$$
- **模型胜率**：$\text{WinRate} = \sigma(r(x, y_w) - r(x, y_l))$。若胜率接近 1.0，说明对齐完全收敛！

In [ ]:
def compute_dpo_implicit_rewards(pi_logps, ref_logps, beta=0.1):
    """根据 DPO 隐式映射公式提取绝对奖励打分"""
    return beta * (pi_logps - ref_logps)

r_impl_w = compute_dpo_implicit_rewards(pi_w, ref_w, beta=0.1)
r_impl_l = compute_dpo_implicit_rewards(pi_l, ref_l, beta=0.1)
win_rate = torch.sigmoid(r_impl_w - r_impl_l).mean()
print("优胜回答隐式奖励:", r_impl_w.numpy())
print("落败回答隐式奖励:", r_impl_l.numpy())
print(f"当前对齐动态胜率: {win_rate.item() * 100:.2f}%")
assert win_rate.item() > 0.5
print(">>> DPO 隐式奖励提取验证成功！")

---
## 模块八：GRPO (群组相对策略优化) 组内均值方差归一化手撕 (DeepSeek 核心)

### 【笔试压轴神级考点】
1. **彻底告别 Critic 价值网络**：传统 PPO 需要额外维护一个参数量庞大的 Critic 网络来估计 Baseline $V(s)$；
2. **组内赛马 (Group Relative)**：针对同一个 Prompt $q$，采样生成 $G$ 个候选回答 $(o_1, o_2, \dots, o_G)$，通过打分模型获得即时奖励 $(R_1, R_2, \dots, R_G)$；
3. **组内优势归一化公式**：
   $$\tilde{A}_i = \frac{R_i - \text{mean}(R)}{\text{std}(R) + \epsilon}$$
   全靠同行衬托！高于平均水平的样本为正优势，低于平均的为负优势。

In [ ]:
def grpo_group_advantages(rewards, eps=1e-8):
    """
    rewards: (B, G) 每个 Prompt 生成 G 个回答的奖励打分
    """
    # 在组内维度 G 计算均值与标准差
    mean = rewards.mean(dim=-1, keepdim=True)
    std = rewards.std(dim=-1, keepdim=True)
    
    # 组内相对优势归一化
    norm_advantages = (rewards - mean) / (std + eps)
    return norm_advantages

# 测试 GRPO 组内赛马归一化
dummy_group_rewards = torch.tensor([[10.0, 20.0, 30.0, 40.0]]) # 1 个 prompt 生成 4 个答案
grpo_adv = grpo_group_advantages(dummy_group_rewards)
print("GRPO 组内归一化相对优势 (均值=0, 方差=1):", grpo_adv.numpy())
assert torch.isclose(grpo_adv.mean(), torch.tensor(0.0), atol=1e-5)
assert grpo_adv[0, -1] > 0 and grpo_adv[0, 0] < 0
print(">>> GRPO 组内赛马归一化验证通过！")

---
## 模块九：GRPO Token-Level 策略损失与相对优势联合更新

### 【笔试考点与公式】
- **GRPO 目标函数**：
  $$\mathcal{L}_{\text{GRPO}}(\theta) = - \frac{1}{G} \sum_{i=1}^G \left( \min(r_{i, t} \tilde{A}_i, \text{clip}(r_{i, t}, 1-\epsilon, 1+\epsilon) \tilde{A}_i) - \beta D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}}) \right)$$

In [ ]:
def grpo_token_loss(log_probs, old_log_probs, ref_log_probs, advantages, beta=0.04, eps_clip=0.2):
    """
    GRPO Token 级联合损失
    log_probs, old_log_probs, ref_log_probs: (G, seq_len)
    advantages: (G, 1) 广播优势
    """
    ratio = torch.exp(log_probs - old_log_probs)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - eps_clip, 1.0 + eps_clip) * advantages
    policy_term = torch.min(surr1, surr2)
    
    # KL 惩罚项
    kl = torch.exp(ref_log_probs - log_probs) - 1.0 - (ref_log_probs - log_probs)
    
    loss = -(policy_term - beta * kl).mean()
    return loss

# 验证 GRPO 联合损失
G, L = 4, 5
advs_g = grpo_adv.view(G, 1) # (4, 1)
lp = torch.randn(G, L)
old_lp = lp.clone()
ref_lp = lp.clone()
grpo_l = grpo_token_loss(lp, old_lp, ref_lp, advs_g)
print("GRPO Token 级损失标量:", grpo_l.item())
assert not torch.isnan(grpo_l)
print(">>> GRPO 端到端损失验证成功！")

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. GAE 逆序递推: delta_t = r + gamma*V_next - V_curr，last_adv = delta + gamma*lam*last_adv 从后往前倒算。
2. PPO-Clip 悲观选小: -min(ratio * A, clip(ratio, 1-eps, 1+eps) * A)，加负号转为下降梯度。
3. Schulman 逆向 KL: ratio - 1 - log(ratio) 恒正且方差极低，告别传统对数差导致的负值崩溃。
4. DPO 免模型闭式解: -logsigmoid(beta * [(pi_w - pi_l) - (ref_w - ref_l)])，直击偏好优化本质。
5. GRPO 告别 Critic: 同一 Prompt 采样 G 组答案，(R - mean) / std 组内相对归一化实现极致降本。
```